# 🎬 YouTube Átirat Letöltő (YouTube Transcript Downloader Pro)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krisatrader/YT-text-downloader/blob/main/youtube_transcript_downloader_colab.ipynb)

Tölts le teljes YouTube lejátszási listákat, csatornákat vagy egyedi videókat betűről betűre **.md** és **.txt** formátumban teljesen ingyen, közvetlenül a Google szupergyors felhőjéből (0 db YouTube 429 blokkolással)!

---

In [ ]:
#@title 🚀 1. Telepítés és Előkészítés (Kattints ide a futtatáshoz ▶️)
#@markdown Ez a lépés letölti a kódot a GitHub-ról és telepíti a szükséges csomagokat.

import os, sys

!rm -rf /content/YT-text-downloader
!git clone https://github.com/krisatrader/YT-text-downloader.git /content/YT-text-downloader
%cd /content/YT-text-downloader
!pip install -q -r requirements.txt

print("\n✅ Sikeres előkészítés! Válassz az alábbi 2/A vagy 2/B futtatási mód közül.")

In [ ]:
#@title ⚡ 2/A. 1-Kattintásos Letöltés Közvetlenül Itt (NINCS SZÜKSÉG KÜLSŐ LINKRE!)
#@markdown **Írd be a YouTube linket az alábbi mezőbe, és kattints a bal oldali ▶️ gombra.** A letöltött ZIP automatikusan megérkezik a gépedre!

YouTube_URL = "https://youtube.com/playlist?list=PLKfy8g4yBtRY&si=IO9L0gnJPgCAurG_" #@param {type:"string"}
Formatum = "both" #@param ["both", "md", "txt"]
Nyelvek = "hu,en" #@param {type:"string"}
Forditas_Celnyelve = "original" #@param ["original", "hu", "en", "de", "es", "fr", "it"]
Idobelyegek = True #@param {type:"boolean"}
Max_Videok = 0 #@param {type:"integer"}
Keses_Masodperc = "1.5-2.5" #@param {type:"string"}

import os, sys, zipfile
from google.colab import files

%cd /content/YT-text-downloader
from transcript_downloader import DownloaderEngine, TranscriptTranslator, sanitize_filename

# Késleltetés feldolgozása
delay_min, delay_max = 1.5, 2.5
if "-" in Keses_Masodperc:
    try:
        dparts = Keses_Masodperc.split("-")
        delay_min, delay_max = float(dparts[0]), float(dparts[1])
    except Exception:
        pass

engine = DownloaderEngine(
    output_dir="transcripts_output",
    output_format=Formatum,
    target_language=Forditas_Celnyelve,
    delay_range=(delay_min, delay_max),
    preferred_languages=[l.strip() for l in Nyelvek.split(",") if l.strip()],
    include_timestamps=Idobelyegek,
    limit=Max_Videok if Max_Videok > 0 else None
)

print(f"🚀 Letöltés indítása a Google felhőszervereiről: {YouTube_URL}...")
res = engine.run(YouTube_URL)

if res and res.get("collection_title"):
    col_title = res["collection_title"]
    clean_title = sanitize_filename(col_title)
    target_dir = os.path.join("transcripts_output", clean_title)
    zip_filename = f"{clean_title}.zip"

    with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, filenames in os.walk(target_dir):
            for fname in filenames:
                fpath = os.path.join(root, fname)
                zf.write(fpath, os.path.relpath(fpath, target_dir))

    print(f"\n📦 ZIP fájl elkészült: {zip_filename}")
    print("⬇️ Letöltés kezdeményezése a böngésződben...")
    files.download(zip_filename)
else:
    print("⚠️ Nem sikerült a gyűjtemény letöltése vagy nincsenek videók.")

In [ ]:
#@title 🌐 2/B. Webes Kezelőfelület Indítása (Opcionális)
#@markdown Ha a szép sötét módú webes kezelőfelületet szeretnéd használni valós idejű élő naplóval.

import os, sys, time, subprocess, re, urllib.request
from google.colab.output import serve_kernel_port_as_window

%cd /content/YT-text-downloader

# 1. Web szerver indítása a háttérben
print("🚀 Web szerver indítása...")
web_proc = subprocess.Popen([sys.executable, "web_app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("\n" + "="*65)
print("🎉 1. OPCIÓ (Google Colab Közvetlen Link):")
try:
    serve_kernel_port_as_window(8000, anchor_text="👉 KATTINTS IDE A WEB FELÜLET MEGNYITÁSÁHOZ (Google Colab)")
except Exception as e:
    print(f"Colab port: {e}")

# 2. Cloudflared letöltése tartalék publikus linkhez
cf_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_path):
    try:
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf_path)
        os.chmod(cf_path, 0o777)
    except Exception:
        pass

if os.path.exists(cf_path):
    tunnel_proc = subprocess.Popen([cf_path, "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)
    for line in tunnel_proc.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\n🎉 2. OPCIÓ (Cloudflare Publikus Link):")
            print(f"👉 {match.group(0)}")
            break
print("="*65 + "\n")

# Szerver életben tartása
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Szerver leállítva.")